In [8]:
# ============================================================
# PHASE 3 — Cell 1 (UPDATED): Load v3 split
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os, json
import numpy as np
import wfdb
from scipy.signal import butter, filtfilt
from collections import Counter

PROJECT_DIR = '/content/drive/MyDrive/ecg-transcovnet'
RAW_DIR = os.path.join(PROJECT_DIR, 'data', 'mitdb')
PROCESSED_DIR = os.path.join(PROJECT_DIR, 'data', 'processed')
SPLITS_DIR = os.path.join(PROJECT_DIR, 'splits')
OUTPUTS_DIR = os.path.join(PROJECT_DIR, 'outputs')

# ⚠️ CHANGE: Load v3 split (with 104 in val)
with open(os.path.join(SPLITS_DIR, 'mitbih_subject_split_v3.json')) as f:
    split_v3 = json.load(f)

train_records = split_v3['train_records']
val_records = split_v3['val_records']
test_records = split_v3['test_records']

print(f"✅ Loaded STRICT subject-disjoint split (v3)")
print(f"   Train: {len(train_records)} records")
print(f"   Val:   {len(val_records)} records")
print(f"   Test:  {len(test_records)} records")

# Load or recreate label config
label_config_path = os.path.join(OUTPUTS_DIR, 'label_config.json')
if os.path.exists(label_config_path):
    with open(label_config_path) as f:
        label_config = json.load(f)
else:
    label_config = {
        'label_protocol': 'NS_V_Q',
        'classes': ['N/S', 'V', 'Q'],
        'keep_classes': [0, 1, 2, 4],
        'label_map': {0: 0, 1: 0, 2: 1, 4: 2},
    }
    with open(label_config_path, 'w') as f:
        json.dump(label_config, f, indent=2)

print(f"\n✅ Label protocol: {label_config['label_protocol']}")
print(f"   Classes: {label_config['classes']}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Loaded STRICT subject-disjoint split (v3)
   Train: 37 records
   Val:   5 records
   Test:  6 records

✅ Label protocol: NS_V_Q
   Classes: ['N/S', 'V', 'Q']


In [3]:
# ============================================================
# CHECK: Where is label_config.json?
# ============================================================
import os

PROJECT_DIR = '/content/drive/MyDrive/ecg-transcovnet'

# Check multiple possible locations
locations = [
    os.path.join(PROJECT_DIR, 'outputs', 'label_config.json'),
    os.path.join(PROJECT_DIR, 'label_config.json'),
    os.path.join(PROJECT_DIR, 'data', 'label_config.json'),
]

print("Searching for label_config.json:")
for path in locations:
    if os.path.exists(path):
        print(f"✅ FOUND: {path}")
    else:
        print(f"❌ NOT FOUND: {path}")

# Check outputs folder
outputs_dir = os.path.join(PROJECT_DIR, 'outputs')
print(f"\nContents of outputs/:")
if os.path.exists(outputs_dir):
    for f in sorted(os.listdir(outputs_dir)):
        print(f"  - {f}")
else:
    print(f"  ❌ outputs/ not found")

Searching for label_config.json:
❌ NOT FOUND: /content/drive/MyDrive/ecg-transcovnet/outputs/label_config.json
❌ NOT FOUND: /content/drive/MyDrive/ecg-transcovnet/label_config.json
❌ NOT FOUND: /content/drive/MyDrive/ecg-transcovnet/data/label_config.json

Contents of outputs/:
  - baseline_cnn_3class_5fold.json
  - baseline_cnn_4class_5fold.json
  - baseline_cnn_5fold_cv.json
  - oof_predictions_best.npy
  - oof_y_true.npy
  - pan_tompkins_hybrid_results.json
  - pan_tompkins_multi_lead.json
  - pan_tompkins_per_record.csv
  - pan_tompkins_single_lead.json


In [9]:
# ============================================================
# RECREATE label_config.json + VERIFY + FORCE SYNC
# ============================================================
import os, json, time

PROJECT_DIR = '/content/drive/MyDrive/ecg-transcovnet'
OUTPUTS_DIR = os.path.join(PROJECT_DIR, 'outputs')
os.makedirs(OUTPUTS_DIR, exist_ok=True)

# Recreate
label_config = {
    'label_protocol': 'NS_V_Q',
    'classes': ['N/S', 'V', 'Q'],
    'keep_classes': [0, 1, 2, 4],
    'label_map': {0: 0, 1: 0, 2: 1, 4: 2},
    'notes': 'S merged with N; clinically valid (both narrow QRS)'
}

config_path = os.path.join(OUTPUTS_DIR, 'label_config.json')
with open(config_path, 'w') as f:
    json.dump(label_config, f, indent=2)

# Force flush to Drive
time.sleep(2)

# Verify immediately
if os.path.exists(config_path):
    with open(config_path) as f:
        loaded = json.load(f)
    size = os.path.getsize(config_path)
    print(f"✅ File exists: {config_path}")
    print(f"   Size: {size} bytes")
    print(f"   Protocol: {loaded['label_protocol']}")
    print(f"   Classes: {loaded['classes']}")
else:
    print(f"❌ File NOT found after write")

# List outputs dir
print(f"\n📁 outputs/ contents:")
for f in sorted(os.listdir(OUTPUTS_DIR)):
    print(f"   - {f}")

✅ File exists: /content/drive/MyDrive/ecg-transcovnet/outputs/label_config.json
   Size: 267 bytes
   Protocol: NS_V_Q
   Classes: ['N/S', 'V', 'Q']

📁 outputs/ contents:
   - baseline_cnn_3class_5fold.json
   - baseline_cnn_4class_5fold.json
   - baseline_cnn_5fold_cv.json
   - label_config.json
   - oof_predictions_best.npy
   - oof_y_true.npy
   - pan_tompkins_hybrid_results.json
   - pan_tompkins_multi_lead.json
   - pan_tompkins_per_record.csv
   - pan_tompkins_single_lead.json


In [10]:
# ============================================================
# PHASE 3 — Cell 2: Define Functions (v3 split)
# ============================================================
AAMI_MAP = {
    'N': 'N', 'L': 'N', 'R': 'N', 'e': 'N', 'j': 'N',
    'A': 'S', 'a': 'S', 'J': 'S', 'S': 'S',
    'V': 'V', 'E': 'V',
    'F': 'F',
    '/': 'Q', 'f': 'Q', 'Q': 'Q'
}
ORIG_TO_IDX = {'N': 0, 'S': 1, 'V': 2, 'F': 3, 'Q': 4}
LABEL_MAP = {0: 0, 1: 0, 2: 1, 4: 2}  # NS→0, V→1, Q→2, F dropped
FINAL_CLASS_NAMES = ['N/S', 'V', 'Q']

FS = 360
PRE_SAMPLES = 99
POST_SAMPLES = 160
BEAT_LEN = PRE_SAMPLES + POST_SAMPLES


def bandpass_filter(signal, fs=FS, low=0.5, high=40.0, order=3):
    nyq = fs / 2.0
    b, a = butter(order, [low/nyq, high/nyq], btype='band')
    return filtfilt(b, a, signal)


def extract_beats_with_rr(record_id, raw_dir=RAW_DIR):
    sig, fields = wfdb.rdsamp(os.path.join(raw_dir, record_id))
    ann = wfdb.rdann(os.path.join(raw_dir, record_id), 'atr')

    lead = sig[:, 0]
    filtered = bandpass_filter(lead)

    r_peaks, orig_classes = [], []
    for sample, sym in zip(ann.sample, ann.symbol):
        cls = AAMI_MAP.get(sym)
        if cls is None:
            continue
        r_peaks.append(sample)
        orig_classes.append(ORIG_TO_IDX[cls])

    r_peaks = np.array(r_peaks)
    orig_classes = np.array(orig_classes)
    n = len(filtered)
    rr_intervals = np.diff(r_peaks)

    X, y, R = [], [], []
    for i, (sample, orig_cls) in enumerate(zip(r_peaks, orig_classes)):
        if orig_cls == 3:
            continue
        new_cls = LABEL_MAP[orig_cls]

        start = sample - PRE_SAMPLES
        end = sample + POST_SAMPLES
        if start < 0 or end > n:
            continue

        beat = filtered[start:end]

        pre_rr = rr_intervals[i-1] if i > 0 else (rr_intervals[0] if len(rr_intervals) > 0 else FS)
        post_rr = rr_intervals[i] if i < len(rr_intervals) else (rr_intervals[-1] if len(rr_intervals) > 0 else FS)
        rr_ratio = pre_rr / (post_rr + 1e-8)
        local_hr = 60.0 * FS / ((pre_rr + post_rr) / 2.0 + 1e-8)

        X.append(beat)
        y.append(new_cls)
        R.append([pre_rr, post_rr, rr_ratio, local_hr])

    return (np.array(X, dtype=np.float32),
            np.array(y, dtype=np.int64),
            np.array(R, dtype=np.float32))


print(f"✅ Functions defined")
print(f"   Classes: {FINAL_CLASS_NAMES}")

✅ Functions defined
   Classes: ['N/S', 'V', 'Q']


In [11]:
# ============================================================
# PHASE 3 — Cell 3: Extract (v3 split)
# ============================================================
def build_split_arrays(record_list, label):
    Xs, ys, Rs, rec_ids = [], [], [], []
    for rec in record_list:
        X, y, R = extract_beats_with_rr(rec)
        Xs.append(X); ys.append(y); Rs.append(R)
        rec_ids.extend([rec] * len(y))
        print(f'  {rec}: {len(y)} beats')
    X_all = np.concatenate(Xs, axis=0)
    y_all = np.concatenate(ys, axis=0)
    R_all = np.concatenate(Rs, axis=0)
    print(f'{label} TOTAL: X={X_all.shape}, R={R_all.shape}')
    print(f'  Labels: {dict(Counter(y_all))}\n')
    return X_all, y_all, R_all, np.array(rec_ids)

print("Extracting TRAIN...")
X_train, y_train, R_train, rec_train = build_split_arrays(train_records, 'TRAIN')

print("Extracting VAL...")
X_val, y_val, R_val, rec_val = build_split_arrays(val_records, 'VAL')

print("Extracting TEST...")
X_test, y_test, R_test, rec_test = build_split_arrays(test_records, 'TEST')

print("=" * 70)
print("EXTRACTION COMPLETE")
print("=" * 70)
print(f"Train: X={X_train.shape}, Labels: {dict(Counter(y_train))}")
print(f"Val:   X={X_val.shape}, Labels: {dict(Counter(y_val))}")
print(f"Test:  X={X_test.shape}, Labels: {dict(Counter(y_test))}")

Extracting TRAIN...
  103: 2083 beats
  105: 2572 beats
  106: 2027 beats
  107: 2136 beats
  109: 2529 beats
  111: 2124 beats
  112: 2538 beats
  116: 2411 beats
  117: 1534 beats
  118: 2277 beats
  119: 1987 beats
  121: 1862 beats
  122: 2474 beats
  123: 1517 beats
  124: 1613 beats
  200: 2598 beats
  201: 1961 beats
  202: 2134 beats
  203: 2979 beats
  205: 2644 beats
  207: 1859 beats
  208: 2581 beats
  209: 3004 beats
  210: 2638 beats
  212: 2747 beats
  213: 2887 beats
  215: 3361 beats
  217: 2208 beats
  220: 2046 beats
  221: 2427 beats
  222: 2481 beats
  223: 2590 beats
  228: 2053 beats
  230: 2255 beats
  231: 1570 beats
  232: 1780 beats
  234: 2753 beats
TRAIN TOTAL: X=(85240, 259), R=(85240, 4)
  Labels: {np.int64(0): 75332, np.int64(1): 6018, np.int64(2): 3890}

Extracting VAL...
  100: 2271 beats
  104: 2227 beats
  113: 1794 beats
  115: 1952 beats
  233: 3066 beats
VAL TOTAL: X=(11310, 259), R=(11310, 4)
  Labels: {np.int64(0): 8415, np.int64(1): 833, np.int

In [12]:
# ============================================================
# PHASE 3 — Cell 4: Normalize + Save (v3)
# ============================================================
def per_beat_normalize(X, eps=1e-8):
    m = X.mean(axis=1, keepdims=True)
    s = X.std(axis=1, keepdims=True)
    return (X - m) / (s + eps)

X_train_n = per_beat_normalize(X_train)
X_val_n = per_beat_normalize(X_val)
X_test_n = per_beat_normalize(X_test)

np.savez_compressed(os.path.join(PROCESSED_DIR, 'train_v2.npz'),
                    X=X_train_n, y=y_train, R=R_train, record_id=rec_train)
np.savez_compressed(os.path.join(PROCESSED_DIR, 'val_v2.npz'),
                    X=X_val_n, y=y_val, R=R_val, record_id=rec_val)
np.savez_compressed(os.path.join(PROCESSED_DIR, 'test_v2.npz'),
                    X=X_test_n, y=y_test, R=R_test, record_id=rec_test)

with open(os.path.join(PROCESSED_DIR, 'preprocessing_config_v2.json'), 'w') as f:
    json.dump({
        'split_file': 'mitbih_subject_split_v3.json',
        'label_protocol': 'NS_V_Q',
        'class_names': FINAL_CLASS_NAMES,
        'pre_samples': PRE_SAMPLES,
        'post_samples': POST_SAMPLES,
        'beat_len': BEAT_LEN,
        'fs': FS,
        'filter': 'Butterworth 0.5-40 Hz order 3',
        'normalization': 'per-beat Z-score',
        'n_rr_features': 4,
        'train_beats': int(len(y_train)),
        'val_beats': int(len(y_val)),
        'test_beats': int(len(y_test))
    }, f, indent=2)

print("✅ Saved:")
print(f"   train_v2.npz: X={X_train_n.shape}, R={R_train.shape}")
print(f"   val_v2.npz:   X={X_val_n.shape}, R={R_val.shape}")
print(f"   test_v2.npz:  X={X_test_n.shape}, R={R_test.shape}")

# Verify
print("\n" + "=" * 70)
print("VERIFICATION")
print("=" * 70)
for f in ['train_v2.npz', 'val_v2.npz', 'test_v2.npz', 'preprocessing_config_v2.json']:
    path = os.path.join(PROCESSED_DIR, f)
    if os.path.exists(path):
        size = os.path.getsize(path) / 1024 / 1024
        print(f"✅ {f}: {size:.2f} MB")
    else:
        print(f"❌ {f}: NOT FOUND")

✅ Saved:
   train_v2.npz: X=(85240, 259), R=(85240, 4)
   val_v2.npz:   X=(11310, 259), R=(11310, 4)
   test_v2.npz:  X=(12097, 259), R=(12097, 4)

VERIFICATION
✅ train_v2.npz: 78.42 MB
✅ val_v2.npz: 10.45 MB
✅ test_v2.npz: 11.19 MB
✅ preprocessing_config_v2.json: 0.00 MB


In [13]:
import os

PROJECT_DIR = '/content/drive/MyDrive/ecg-transcovnet'
PROCESSED_DIR = os.path.join(PROJECT_DIR, 'data', 'processed')
SPLITS_DIR = os.path.join(PROJECT_DIR, 'splits')
OUTPUTS_DIR = os.path.join(PROJECT_DIR, 'outputs')

print("=" * 70)
print("FINAL VERIFICATION — Phase 3 Artifacts")
print("=" * 70)

print("\n📁 data/processed/:")
for f in ['train_v2.npz', 'val_v2.npz', 'test_v2.npz', 'preprocessing_config_v2.json']:
    path = os.path.join(PROCESSED_DIR, f)
    if os.path.exists(path):
        size = os.path.getsize(path) / 1024 / 1024
        print(f"   ✅ {f}: {size:.2f} MB")
    else:
        print(f"   ❌ {f}: MISSING")

print("\n📁 splits/:")
for f in ['mitbih_subject_split_v2.json', 'mitbih_subject_split_v3.json']:
    path = os.path.join(SPLITS_DIR, f)
    if os.path.exists(path):
        print(f"   ✅ {f}")
    else:
        print(f"   ❌ {f}: MISSING")

print("\n📁 outputs/:")
for f in ['label_config.json']:
    path = os.path.join(OUTPUTS_DIR, f)
    if os.path.exists(path):
        print(f"   ✅ {f}")
    else:
        print(f"   ❌ {f}: MISSING")

print("\n📁 audit/:")
audit_dir = os.path.join(PROJECT_DIR, 'audit')
if os.path.exists(audit_dir):
    for f in sorted(os.listdir(audit_dir)):
        print(f"   ✅ {f}")

print("\n" + "=" * 70)
print("✅ All Phase 3 artifacts saved to Drive")
print("=" * 70)

FINAL VERIFICATION — Phase 3 Artifacts

📁 data/processed/:
   ✅ train_v2.npz: 78.42 MB
   ✅ val_v2.npz: 10.45 MB
   ✅ test_v2.npz: 11.19 MB
   ✅ preprocessing_config_v2.json: 0.00 MB

📁 splits/:
   ✅ mitbih_subject_split_v2.json
   ✅ mitbih_subject_split_v3.json

📁 outputs/:
   ✅ label_config.json

📁 audit/:
   ✅ CURRENT_PIPELINE_AUDIT.md
   ✅ label_audit.json
   ✅ leakage_audit.json
   ✅ subject_mapping.json

✅ All Phase 3 artifacts saved to Drive


In [4]:
# ============================================================
# RECREATE: label_config.json
# ============================================================
import os, json

PROJECT_DIR = '/content/drive/MyDrive/ecg-transcovnet'
OUTPUTS_DIR = os.path.join(PROJECT_DIR, 'outputs')
os.makedirs(OUTPUTS_DIR, exist_ok=True)

# Recreate NS_V_Q label config
label_config = {
    'label_protocol': 'NS_V_Q',
    'classes': ['N/S', 'V', 'Q'],
    'keep_classes': [0, 1, 2, 4],
    'label_map': {0: 0, 1: 0, 2: 1, 4: 2},
    'notes': 'S merged with N; clinically valid (both narrow QRS)',
    'created_at': 'Phase 3'
}

config_path = os.path.join(OUTPUTS_DIR, 'label_config.json')
with open(config_path, 'w') as f:
    json.dump(label_config, f, indent=2)

print(f"✅ Recreated: {config_path}")
print(f"   Protocol: {label_config['label_protocol']}")
print(f"   Classes: {label_config['classes']}")

# Verify
with open(config_path) as f:
    loaded = json.load(f)
print(f"\n✅ Verified: {loaded['label_protocol']}")

✅ Recreated: /content/drive/MyDrive/ecg-transcovnet/outputs/label_config.json
   Protocol: NS_V_Q
   Classes: ['N/S', 'V', 'Q']

✅ Verified: NS_V_Q


In [5]:
# ============================================================
# PHASE 3 — Cell 2: Define Functions
# ============================================================
AAMI_MAP = {
    'N': 'N', 'L': 'N', 'R': 'N', 'e': 'N', 'j': 'N',
    'A': 'S', 'a': 'S', 'J': 'S', 'S': 'S',
    'V': 'V', 'E': 'V',
    'F': 'F',
    '/': 'Q', 'f': 'Q', 'Q': 'Q'
}
ORIG_TO_IDX = {'N': 0, 'S': 1, 'V': 2, 'F': 3, 'Q': 4}

# NS_V_Q label mapping:
# N (0) → 0 (N/S)
# S (1) → 0 (N/S)
# V (2) → 1 (V)
# F (3) → DROP
# Q (4) → 2 (Q)
LABEL_MAP = {0: 0, 1: 0, 2: 1, 4: 2}
FINAL_CLASS_NAMES = ['N/S', 'V', 'Q']

FS = 360
PRE_SAMPLES = 99
POST_SAMPLES = 160
BEAT_LEN = PRE_SAMPLES + POST_SAMPLES


def bandpass_filter(signal, fs=FS, low=0.5, high=40.0, order=3):
    """Butterworth bandpass 0.5-40 Hz — fixed design, no leakage."""
    nyq = fs / 2.0
    b, a = butter(order, [low/nyq, high/nyq], btype='band')
    return filtfilt(b, a, signal)


def extract_beats_with_rr(record_id, raw_dir=RAW_DIR):
    """Extract beats + 4 RR features with NS_V_Q labels."""
    sig, fields = wfdb.rdsamp(os.path.join(raw_dir, record_id))
    ann = wfdb.rdann(os.path.join(raw_dir, record_id), 'atr')

    lead = sig[:, 0]  # MLII
    filtered = bandpass_filter(lead)

    r_peaks, orig_classes = [], []
    for sample, sym in zip(ann.sample, ann.symbol):
        cls = AAMI_MAP.get(sym)
        if cls is None:
            continue
        r_peaks.append(sample)
        orig_classes.append(ORIG_TO_IDX[cls])

    r_peaks = np.array(r_peaks)
    orig_classes = np.array(orig_classes)
    n = len(filtered)
    rr_intervals = np.diff(r_peaks)

    X, y, R = [], [], []
    for i, (sample, orig_cls) in enumerate(zip(r_peaks, orig_classes)):
        # Drop F-class (label 3)
        if orig_cls == 3:
            continue

        # Apply NS_V_Q mapping
        new_cls = LABEL_MAP[orig_cls]

        start = sample - PRE_SAMPLES
        end = sample + POST_SAMPLES
        if start < 0 or end > n:
            continue

        beat = filtered[start:end]

        # RR features
        pre_rr = rr_intervals[i-1] if i > 0 else (rr_intervals[0] if len(rr_intervals) > 0 else FS)
        post_rr = rr_intervals[i] if i < len(rr_intervals) else (rr_intervals[-1] if len(rr_intervals) > 0 else FS)
        rr_ratio = pre_rr / (post_rr + 1e-8)
        local_hr = 60.0 * FS / ((pre_rr + post_rr) / 2.0 + 1e-8)

        X.append(beat)
        y.append(new_cls)
        R.append([pre_rr, post_rr, rr_ratio, local_hr])

    return (np.array(X, dtype=np.float32),
            np.array(y, dtype=np.int64),
            np.array(R, dtype=np.float32))


print(f"✅ Functions defined")
print(f"   Classes: {FINAL_CLASS_NAMES}")
print(f"   BEAT_LEN: {BEAT_LEN}")
print(f"   Pre/Post samples: {PRE_SAMPLES}/{POST_SAMPLES}")

✅ Functions defined
   Classes: ['N/S', 'V', 'Q']
   BEAT_LEN: 259
   Pre/Post samples: 99/160


In [6]:
# ============================================================
# PHASE 3 — Cell 3: Extract Beats + RR for All Splits
# ============================================================
def build_split_arrays(record_list, label):
    Xs, ys, Rs, rec_ids = [], [], [], []
    for rec in record_list:
        X, y, R = extract_beats_with_rr(rec)
        Xs.append(X); ys.append(y); Rs.append(R)
        rec_ids.extend([rec] * len(y))
        print(f'  {rec}: {len(y)} beats')
    X_all = np.concatenate(Xs, axis=0)
    y_all = np.concatenate(ys, axis=0)
    R_all = np.concatenate(Rs, axis=0)
    print(f'{label} TOTAL: X={X_all.shape}, R={R_all.shape}')
    print(f'  Labels: {dict(Counter(y_all))}\n')
    return X_all, y_all, R_all, np.array(rec_ids)


print("=" * 70)
print("EXTRACTING BEATS + RR FEATURES")
print("=" * 70)

print("\nExtracting TRAIN...")
X_train, y_train, R_train, rec_train = build_split_arrays(train_records, 'TRAIN')

print("Extracting VAL...")
X_val, y_val, R_val, rec_val = build_split_arrays(val_records, 'VAL')

print("Extracting TEST...")
X_test, y_test, R_test, rec_test = build_split_arrays(test_records, 'TEST')

print("=" * 70)
print("EXTRACTION COMPLETE")
print("=" * 70)
print(f"Train: X={X_train.shape}, R={R_train.shape}")
print(f"Val:   X={X_val.shape}, R={R_val.shape}")
print(f"Test:  X={X_test.shape}, R={R_test.shape}")
print(f"\nClass distribution:")
print(f"Train: {dict(Counter(y_train))}")
print(f"Val:   {dict(Counter(y_val))}")
print(f"Test:  {dict(Counter(y_test))}")

EXTRACTING BEATS + RR FEATURES

Extracting TRAIN...
  103: 2083 beats
  104: 2227 beats
  105: 2572 beats
  106: 2027 beats
  107: 2136 beats
  109: 2529 beats
  111: 2124 beats
  112: 2538 beats
  116: 2411 beats
  117: 1534 beats
  118: 2277 beats
  119: 1987 beats
  121: 1862 beats
  122: 2474 beats
  123: 1517 beats
  124: 1613 beats
  200: 2598 beats
  201: 1961 beats
  202: 2134 beats
  203: 2979 beats
  205: 2644 beats
  207: 1859 beats
  208: 2581 beats
  209: 3004 beats
  210: 2638 beats
  212: 2747 beats
  213: 2887 beats
  215: 3361 beats
  217: 2208 beats
  220: 2046 beats
  221: 2427 beats
  222: 2481 beats
  223: 2590 beats
  228: 2053 beats
  230: 2255 beats
  231: 1570 beats
  232: 1780 beats
  234: 2753 beats
TRAIN TOTAL: X=(87467, 259), R=(87467, 4)
  Labels: {np.int64(0): 75495, np.int64(2): 5952, np.int64(1): 6020}

Extracting VAL...
  100: 2271 beats
  113: 1794 beats
  115: 1952 beats
  233: 3066 beats
VAL TOTAL: X=(9083, 259), R=(9083, 4)
  Labels: {np.int64(0): 

In [7]:
# ============================================================
# ADJUST SPLIT: Move record 104 to Val (Q-heavy)
# ============================================================
import os, json

PROJECT_DIR = '/content/drive/MyDrive/ecg-transcovnet'
SPLITS_DIR = os.path.join(PROJECT_DIR, 'splits')
AUDIT_DIR = os.path.join(PROJECT_DIR, 'audit')

# Load subject mapping
with open(os.path.join(AUDIT_DIR, 'subject_mapping.json')) as f:
    mapping = json.load(f)

record_to_subject = mapping['record_to_subject']
subject_to_records = mapping['subject_to_records']

# Load current v2 split
with open(os.path.join(SPLITS_DIR, 'mitbih_subject_split_v2.json')) as f:
    split_v2 = json.load(f)

# Get current splits
train_records = list(split_v2['train_records'])
val_records = list(split_v2['val_records'])
test_records = list(split_v2['test_records'])

# ============================================================
# MOVE: Record 104 (subject S005) from Train → Val
# ============================================================
RECORD_TO_MOVE = '104'
subject_of_record = record_to_subject[RECORD_TO_MOVE]

print(f"Moving record {RECORD_TO_MOVE} (subject {subject_of_record})")
print(f"  From: Train")
print(f"  To:   Val")

# Check if subject has other records
other_records_same_subject = [r for r in subject_to_records[subject_of_record]
                                if r != RECORD_TO_MOVE]
print(f"  Other records from same subject: {other_records_same_subject}")

# Move all records of that subject
records_to_move = subject_to_records[subject_of_record]

# Remove from train
train_records = [r for r in train_records if r not in records_to_move]
# Add to val
val_records = sorted(val_records + records_to_move)

# ============================================================
# Verify no leakage
# ============================================================
train_subjects = set(record_to_subject[r] for r in train_records)
val_subjects = set(record_to_subject[r] for r in val_records)
test_subjects = set(record_to_subject[r] for r in test_records)

assert train_subjects.isdisjoint(val_subjects), "Train ∩ Val not empty"
assert train_subjects.isdisjoint(test_subjects), "Train ∩ Test not empty"
assert val_subjects.isdisjoint(test_subjects), "Val ∩ Test not empty"
print("\n✅ NO SUBJECT LEAKAGE")

# ============================================================
# Save new split
# ============================================================
split_v3 = {
    'protocol': 'STRICT_SUBJECT_DISJOINT_V3',
    'random_seed': 42,
    'split_strategy': 'subject-level with Q-class distributed to val',
    'train_subjects': sorted(train_subjects),
    'val_subjects': sorted(val_subjects),
    'test_subjects': sorted(test_subjects),
    'train_records': sorted(train_records),
    'val_records': sorted(val_records),
    'test_records': sorted(test_records),
    'moved_for_q_distribution': records_to_move,
    'verified_no_leakage': True
}

split_path_v3 = os.path.join(SPLITS_DIR, 'mitbih_subject_split_v3.json')
with open(split_path_v3, 'w') as f:
    json.dump(split_v3, f, indent=2)

print(f"\n✅ New split saved: {split_path_v3}")

# ============================================================
# Summary
# ============================================================
print("\n" + "=" * 70)
print("UPDATED SPLIT SUMMARY")
print("=" * 70)
print(f"\nTrain records ({len(train_records)}): {sorted(train_records)}")
print(f"Val records   ({len(val_records)}): {sorted(val_records)}")
print(f"Test records  ({len(test_records)}): {sorted(test_records)}")
print(f"\n✅ Moved to val: {records_to_move}")

Moving record 104 (subject S005)
  From: Train
  To:   Val
  Other records from same subject: []

✅ NO SUBJECT LEAKAGE

✅ New split saved: /content/drive/MyDrive/ecg-transcovnet/splits/mitbih_subject_split_v3.json

UPDATED SPLIT SUMMARY

Train records (37): ['103', '105', '106', '107', '109', '111', '112', '116', '117', '118', '119', '121', '122', '123', '124', '200', '201', '202', '203', '205', '207', '208', '209', '210', '212', '213', '215', '217', '220', '221', '222', '223', '228', '230', '231', '232', '234']
Val records   (5): ['100', '104', '113', '115', '233']
Test records  (6): ['101', '102', '108', '114', '214', '219']

✅ Moved to val: ['104']
